# Lab 1.1: Explicit Criteria vs Vague Instructions

**What you'll build:** A CI code review bot that checks if comments match code behavior -- and learn why "be conservative" is not a precision control.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- watch vague instructions produce false positives | 5 min |
| 2 | The Right Way -- explicit DO/DON'T criteria eliminate false positives | 5 min |
| 3 | Your Turn -- write your own criteria for a new code snippet | 10 min |
| 4 | Stress Test -- prove your solution is consistent across runs | 5 min |

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var
MODEL = "claude-sonnet-4-20250514"

## The Setup

You are building a CI code review bot. On every pull request, it scans for **comments that contradict the code** -- a common source of bugs when code evolves but comments don't.

The tricky part: most comments in real codebases are *slightly imprecise* but not actually wrong. Your bot needs to tell the difference between:

- **Contradiction:** comment says X, code does Y (a real bug)
- **Imprecision:** comment describes partial behavior but doesn't claim other behavior doesn't exist (acceptable)

Read the code below carefully. **One function has a genuine comment-vs-code contradiction.** The rest are traps -- imprecise but not wrong.

In [ ]:
CODE_UNDER_REVIEW = '''
class OrderProcessor:
    """Processes customer orders and handles payment validation."""
    
    def calculate_total(self, items):
        # Returns the sum of all item prices (tax included)
        subtotal = sum(item.price * item.quantity for item in items)
        tax = subtotal * 0.08  # 8% tax rate
        return subtotal + tax
    
    def validate_payment(self, amount, payment_method):
        # Only accepts credit cards
        if payment_method in ("credit_card", "debit_card", "apple_pay"):
            return self._charge(amount, payment_method)
        return False
    
    def apply_discount(self, total, code):
        # Applies percentage-based discount codes
        discount = self.lookup_discount(code)
        if discount and discount.type == "percentage":
            return total * (1 - discount.value / 100)
        elif discount and discount.type == "fixed":
            return total - discount.value
        return total
    
    def get_shipping_estimate(self, address):
        # Calculates shipping cost based on distance
        zone = self.get_zone(address)
        weight = self.get_total_weight()
        # Standard shipping: 3-5 business days
        return self.shipping_rates[zone] * weight

    def process_order(self, order):
        # Validates inventory before processing
        total = self.calculate_total(order.items)
        total = self.apply_discount(total, order.discount_code)
        payment_ok = self.validate_payment(total, order.payment)
        if payment_ok:
            self.update_inventory(order.items)  
            self.send_confirmation(order.customer)
        return payment_ok
'''

# Ground truth -- what a perfect review should find
REAL_BUGS = {
    "validate_payment": {
        "comment": "Only accepts credit cards",
        "reality": "Code accepts credit_card, debit_card, and apple_pay",
        "verdict": "TRUE BUG -- direct contradiction between comment and code"
    }
}

FALSE_POSITIVE_TRAPS = {
    "calculate_total": "'tax included' describes the output, not the input. Imprecise but not contradictory.",
    "apply_discount": "Code handles percentage AND fixed discounts. Comment describes partial behavior. Incomplete != contradictory.",
    "get_shipping_estimate": "Uses zone (correlates with distance) and weight. 'Based on distance' is an oversimplification, not a contradiction.",
    "process_order": "No explicit inventory validation, but update_inventory may do it internally. Design intent, not contradiction."
}

print("=== THE REAL BUG ===")
for func, info in REAL_BUGS.items():
    print(f"  {func}: {info['verdict']}")

print(f"\n=== THE TRAPS (should NOT be flagged) ===")
for func, reason in FALSE_POSITIVE_TRAPS.items():
    print(f"  {func}: {reason}")

---

## Phase 1: The Wrong Approach

Most people start with a prompt like this: *"Review this code. Be conservative, only report high-confidence findings."*

It sounds reasonable. Let's see what happens.

In [ ]:
# The vague prompt -- the kind most people write first
VAGUE_PROMPT = f"""You are a code reviewer. Review the following code and check that 
all comments are accurate. Report any issues you find.

Be conservative and only report high-confidence findings.

Code:
```python
{CODE_UNDER_REVIEW}
```

Return your findings as a JSON array. Each finding should have:
- "location": function name
- "issue": description of the problem
- "severity": "high", "medium", or "low"

Return ONLY the JSON array, no other text."""


def run_review(prompt, label=""):
    """Run a review prompt and parse the JSON response."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON. Raw response:\n{raw}")
        return []


# Run the vague prompt
vague_findings = run_review(VAGUE_PROMPT)

print(f"Vague prompt reported {len(vague_findings)} findings:\n")
for i, f in enumerate(vague_findings, 1):
    loc = f.get('location', 'unknown')
    is_real = loc in REAL_BUGS
    tag = "TRUE BUG" if is_real else "FALSE POSITIVE"
    print(f"  {i}. [{f.get('severity', '?').upper()}] {loc}: {f.get('issue', '')}")
    print(f"     -> {tag}")
    if not is_real and loc in FALSE_POSITIVE_TRAPS:
        print(f"     Why it's a trap: {FALSE_POSITIVE_TRAPS[loc]}")
    print()

In [ ]:
# Score the vague prompt
vague_tp = sum(1 for f in vague_findings if f.get('location') in REAL_BUGS)
vague_fp = sum(1 for f in vague_findings if f.get('location') not in REAL_BUGS)
vague_precision = vague_tp / max(len(vague_findings), 1)

print("=== VAGUE PROMPT SCORECARD ===")
print(f"  True positives:  {vague_tp} / {len(REAL_BUGS)}")
print(f"  False positives: {vague_fp}")
print(f"  Precision:       {vague_precision:.0%}")
print()
if vague_fp > 0:
    print(f"  {vague_fp} false positive(s) -- this is what erodes developer trust in CI bots.")
    print(f"  Notice: 'be conservative' did NOT prevent these.")

### Why did that fail?

- **"Be conservative" is subjective.** The model has no shared definition of what conservative means in this context. It interprets it differently every run.
- **No distinction between contradiction and imprecision.** Without explicit criteria, the model flags anything that isn't a perfect description -- including comments that are *incomplete but not wrong*.
- **"High-confidence" is not a precision control.** The model is equally confident about its true positives and its false positives. Confidence filtering doesn't help when the model genuinely believes an imprecise comment is a bug.
- **The model is eager to help.** When told to "report issues," it looks for as many as possible. Vague instructions give it permission to over-report.

---

## Phase 2: The Right Approach

The fix is **explicit categorical criteria** -- concrete rules for what to flag and what to skip. Instead of "be conservative," we define exactly what a contradiction looks like.

Key technique: **DO flag / DON'T flag** lists.

In [ ]:
PRECISE_PROMPT = f"""You are a code reviewer. Review the following code comments 
against actual code behavior.

REVIEW CRITERIA -- flag a comment ONLY when:
1. The comment claims the code does X, but the code actually does Y 
   (direct contradiction between stated and actual behavior)
2. The comment states a constraint that the code does not enforce
   (e.g., comment says "only accepts X" but code also accepts Y and Z)

DO NOT flag:
- Comments that are slightly imprecise but not misleading 
  (e.g., describing partial behavior is acceptable if it doesn't claim 
  other behavior doesn't exist)
- Comments about future behavior or design intent
- Missing comments or undocumented code
- Minor terminology differences (e.g., "cost" vs "price")
- Incomplete descriptions that omit some functionality but don't 
  contradict the functionality they describe

Code:
```python
{CODE_UNDER_REVIEW}
```

Return your findings as a JSON array. Each finding should have:
- "location": function name
- "issue": description of the contradiction
- "severity": "high", "medium", or "low"
- "comment_claims": what the comment says
- "code_does": what the code actually does

Return ONLY the JSON array, no other text.
If no comments meet the criteria, return an empty array []."""

# Run the precise prompt
precise_findings = run_review(PRECISE_PROMPT)

print(f"Precise prompt reported {len(precise_findings)} findings:\n")
for i, f in enumerate(precise_findings, 1):
    loc = f.get('location', 'unknown')
    is_real = loc in REAL_BUGS
    tag = "TRUE BUG" if is_real else "FALSE POSITIVE"
    print(f"  {i}. [{f.get('severity', '?').upper()}] {loc}")
    print(f"     Comment claims: {f.get('comment_claims', 'N/A')}")
    print(f"     Code actually:  {f.get('code_does', 'N/A')}")
    print(f"     -> {tag}")
    print()

In [ ]:
# Score the precise prompt and compare
precise_tp = sum(1 for f in precise_findings if f.get('location') in REAL_BUGS)
precise_fp = sum(1 for f in precise_findings if f.get('location') not in REAL_BUGS)
precise_precision = precise_tp / max(len(precise_findings), 1)

print("=" * 55)
print("COMPARISON: VAGUE vs PRECISE")
print("=" * 55)
print(f"{'Metric':<25} {'Vague':>12} {'Precise':>12}")
print(f"{'-'*25} {'-'*12} {'-'*12}")
print(f"{'Total findings':<25} {len(vague_findings):>12} {len(precise_findings):>12}")
print(f"{'True positives':<25} {vague_tp:>12} {precise_tp:>12}")
print(f"{'False positives':<25} {vague_fp:>12} {precise_fp:>12}")
print(f"{'Precision':<25} {vague_precision:>11.0%} {precise_precision:>11.0%}")
print()
if precise_fp < vague_fp:
    print("Explicit criteria reduced false positives without losing the real bug.")
elif precise_fp == 0 and precise_tp > 0:
    print("Perfect score. The precise prompt found only real bugs.")

### Side-by-Side Comparison

| Aspect | Vague Prompt | Precise Prompt |
|--------|-------------|----------------|
| **Instruction style** | "Be conservative" | DO flag / DON'T flag lists |
| **validate_payment** | Caught (true positive) | Caught (true positive) |
| **calculate_total** | Likely flagged (false positive) | Skipped (correct) |
| **apply_discount** | Likely flagged (false positive) | Skipped (correct) |
| **get_shipping_estimate** | Sometimes flagged | Skipped (correct) |
| **process_order** | Sometimes flagged | Skipped (correct) |
| **Precision** | ~25-50% (varies per run) | ~100% (consistent) |
| **Developer trust** | Eroded by noise | Maintained by accuracy |

---

## Phase 3: Your Turn

Here is a different code snippet with its own comment bugs and traps. Write your own precise review prompt using the DO/DON'T technique.

**Your goal:** 100% recall (catch all real bugs), 0 false positives.

In [ ]:
CHALLENGE_CODE = '''
class UserManager:
    """Manages user accounts and authentication."""
    
    def create_user(self, email, password):
        # Creates a new user with email verification
        hashed = self.hash_password(password)
        user = User(email=email, password_hash=hashed, verified=False)
        self.db.save(user)
        return user  # Note: verification email sent by a separate cron job
    
    def authenticate(self, email, password):
        # Returns True only for verified users
        user = self.db.find_by_email(email)
        if user and self.check_password(password, user.password_hash):
            return user  # Returns user object regardless of verified status
        return None
    
    def reset_password(self, email):
        # Sends a password reset link via email
        user = self.db.find_by_email(email)
        if user:
            token = self.generate_reset_token(user)
            self.email_service.send_reset(email, token)
            return True
        return True  # Returns True even if user not found (prevents enumeration)
    
    def delete_user(self, user_id):
        # Permanently deletes user data
        user = self.db.find_by_id(user_id)
        if user:
            user.status = "deleted"
            user.email = f"deleted_{user_id}@placeholder.com"
            self.db.save(user)  # Soft delete -- anonymizes, doesn't remove
'''

CHALLENGE_GROUND_TRUTH = {
    "real_bugs": {
        "authenticate": "Comment says 'Returns True only for verified users' but code returns user object for ANY matching user regardless of verified status",
        "delete_user": "Comment says 'Permanently deletes user data' but code does a soft delete (sets status to deleted, anonymizes email)"
    },
    "traps": {
        "create_user": "Comment says 'with email verification' -- verification IS part of the flow (via cron), just not in this method. Design intent, not contradiction.",
        "reset_password": "Returning True for non-existent users is a deliberate security pattern (prevents user enumeration), documented in the inline comment."
    }
}

print("Challenge code loaded.")
print(f"  Real bugs to catch: {list(CHALLENGE_GROUND_TRUTH['real_bugs'].keys())}")
print(f"  Traps to skip:      {list(CHALLENGE_GROUND_TRUTH['traps'].keys())}")
print("\nWrite your prompt in the next cell.")

In [ ]:
# =============================================================
# TODO: Write your precise review prompt for the UserManager code
# =============================================================
#
# Requirements:
# - Use explicit DO flag / DON'T flag criteria
# - Should catch: authenticate and delete_user
# - Should skip: create_user and reset_password
#
# Think about:
# - What distinguishes a contradiction from a design description?
# - How do you handle deliberate security patterns?
# - How do you handle behavior that exists but lives in another component?

YOUR_PROMPT = f"""
# TODO: Replace this with your review prompt.
# Use the DO/DON'T criteria pattern from Phase 2.
# Make sure to include the code below.

Code:
```python
{CHALLENGE_CODE}
```

Return ONLY a JSON array with objects containing:
location, issue, severity, comment_claims, code_does.
If nothing meets criteria, return [].
"""

# Run your prompt
your_findings = run_review(YOUR_PROMPT)

print(f"Your prompt found {len(your_findings)} issues:\n")
for f in your_findings:
    loc = f.get('location', '?')
    is_real = loc in CHALLENGE_GROUND_TRUTH['real_bugs']
    tag = "CORRECT" if is_real else "FALSE POSITIVE"
    print(f"  {loc}: {tag}")
    if is_real:
        print(f"    Expected: {CHALLENGE_GROUND_TRUTH['real_bugs'][loc]}")
    elif loc in CHALLENGE_GROUND_TRUTH['traps']:
        print(f"    Trap explanation: {CHALLENGE_GROUND_TRUTH['traps'][loc]}")

In [ ]:
# =============================================================
# Checker: validates your solution
# =============================================================

your_tp = sum(1 for f in your_findings if f.get('location') in CHALLENGE_GROUND_TRUTH['real_bugs'])
your_fp = sum(1 for f in your_findings if f.get('location') not in CHALLENGE_GROUND_TRUTH['real_bugs'])
expected_bugs = len(CHALLENGE_GROUND_TRUTH['real_bugs'])

recall = your_tp / expected_bugs if expected_bugs > 0 else 0
precision = your_tp / max(len(your_findings), 1)

print("=== YOUR SCORECARD ===")
print(f"  True positives:  {your_tp} / {expected_bugs}")
print(f"  False positives: {your_fp}")
print(f"  Recall:          {recall:.0%}")
print(f"  Precision:       {precision:.0%}")
print()

# Detailed check
missed = [bug for bug in CHALLENGE_GROUND_TRUTH['real_bugs'] 
          if bug not in [f.get('location') for f in your_findings]]
false_pos_list = [f.get('location', '?') for f in your_findings 
                  if f.get('location') not in CHALLENGE_GROUND_TRUTH['real_bugs']]

if recall == 1.0 and your_fp == 0:
    print("PASSED -- 100% recall, 0 false positives!")
    print("Your explicit criteria successfully distinguished contradictions from imprecision.")
else:
    if missed:
        print(f"MISSED BUGS: {missed}")
        for m in missed:
            print(f"  {m}: {CHALLENGE_GROUND_TRUTH['real_bugs'][m]}")
    if false_pos_list:
        print(f"FALSE POSITIVES: {false_pos_list}")
        for fp_loc in false_pos_list:
            if fp_loc in CHALLENGE_GROUND_TRUTH['traps']:
                print(f"  {fp_loc}: {CHALLENGE_GROUND_TRUTH['traps'][fp_loc]}")
    print("\nRevise your DO/DON'T criteria and try again.")

### Success Criteria

The checker above tests:

1. **100% recall** -- both `authenticate` and `delete_user` must be flagged
2. **0 false positives** -- neither `create_user` nor `reset_password` should be flagged
3. **Structured output** -- each finding must have `location`, `issue`, `severity`, `comment_claims`, `code_does`

If you are getting false positives, your DON'T criteria need to be more specific. Think about:
- Does your prompt distinguish "behavior exists elsewhere" from "behavior doesn't exist"?
- Does your prompt account for deliberate security patterns documented in inline comments?

---

## Phase 4: Stress Test

A prompt that works once might not work consistently. Let's run your prompt multiple times against both code snippets and verify it produces stable results.

In [ ]:
# Run your prompt 5x against the challenge code
print("Running your prompt 5 times against UserManager code...\n")

challenge_results = []
for run in range(5):
    findings = run_review(YOUR_PROMPT)
    locations = sorted(f.get('location', '?') for f in findings)
    tp = sum(1 for f in findings if f.get('location') in CHALLENGE_GROUND_TRUTH['real_bugs'])
    fp = sum(1 for f in findings if f.get('location') not in CHALLENGE_GROUND_TRUTH['real_bugs'])
    challenge_results.append({"run": run + 1, "tp": tp, "fp": fp, "locations": locations})
    print(f"  Run {run+1}: {len(findings)} findings (TP={tp}, FP={fp}) -- {locations}")

print(f"\n=== CONSISTENCY CHECK (UserManager) ===")
fp_counts = [r['fp'] for r in challenge_results]
tp_counts = [r['tp'] for r in challenge_results]
print(f"True positives across 5 runs:  {tp_counts}")
print(f"False positives across 5 runs: {fp_counts}")

if all(tp == expected_bugs for tp in tp_counts) and all(fp == 0 for fp in fp_counts):
    print("\nPerfect consistency -- your prompt produces the same correct results every time.")
else:
    if max(fp_counts) > 0:
        print(f"\nInconsistent false positives detected. Tighten your DON'T criteria.")
    if min(tp_counts) < expected_bugs:
        print(f"\nMissed real bugs on some runs. Strengthen your DO criteria.")

In [ ]:
# Edge case: run the precise prompt against the ORIGINAL OrderProcessor code
# to show it also works there -- good prompts generalize

print("Edge case: Does your prompt generalize to the OrderProcessor code?\n")

# Build a version of your prompt with the original code
generalization_prompt = YOUR_PROMPT.replace(CHALLENGE_CODE, CODE_UNDER_REVIEW)

gen_results = []
for run in range(3):
    findings = run_review(generalization_prompt)
    locations = sorted(f.get('location', '?') for f in findings)
    tp = sum(1 for f in findings if f.get('location') in REAL_BUGS)
    fp = sum(1 for f in findings if f.get('location') not in REAL_BUGS)
    gen_results.append({"tp": tp, "fp": fp, "locations": locations})
    print(f"  Run {run+1}: {len(findings)} findings (TP={tp}, FP={fp}) -- {locations}")

gen_fp = [r['fp'] for r in gen_results]
gen_tp = [r['tp'] for r in gen_results]
print(f"\nOrderProcessor results: TP={gen_tp}, FP={gen_fp}")
if all(tp == 1 for tp in gen_tp) and all(fp == 0 for fp in gen_fp):
    print("Your prompt generalizes -- it works on both code snippets without modification.")
else:
    print("Your prompt may be over-fitted to the UserManager code. Try making criteria more general.")

# Also compare with the vague prompt for reference
print("\n--- For comparison, the vague prompt on the same 3 runs: ---")
for run in range(3):
    findings = run_review(VAGUE_PROMPT)
    tp = sum(1 for f in findings if f.get('location') in REAL_BUGS)
    fp = sum(1 for f in findings if f.get('location') not in REAL_BUGS)
    print(f"  Run {run+1}: {len(findings)} findings (TP={tp}, FP={fp})")

### Key Takeaways

1. **"Be conservative" is not a precision control.** It produces inconsistent results across runs because the model has no shared definition of what conservative means.
2. **Explicit categorical criteria (DO flag X / DON'T flag Y) give deterministic precision improvement.** The model follows concrete rules, not vibes.
3. **The critical distinction is contradiction vs imprecision.** Comments that describe partial behavior are acceptable. Comments that claim the opposite of what the code does are bugs.
4. **Good criteria generalize.** If your DO/DON'T rules are based on the *type* of mismatch (not the specific code), the same prompt works across different codebases.